# SANSA: train the Tamil model on your own recordings

**Steps (one time):**
1. Top menu: **Runtime → Change runtime type → T4 GPU → Save**
2. Top menu: **Runtime → Run all**
3. When it asks, choose the file **sansa_training.zip**
4. When it asks for a Hugging Face token, paste your **write** token from https://huggingface.co/settings/tokens

Takes about 30 to 60 minutes. At the end it prints accuracy **before** and **after** training, and saves the new model to your Hugging Face account as `whisper-tamil-medium-sansa` (private). Then tell Claude it is done.

In [ ]:
# 1. Check the GPU and install tools
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
print(gpu or "NO GPU! Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.")
assert gpu, "Turn on the T4 GPU first"
!pip install -q "transformers==4.46.3" "datasets==3.1.0" "accelerate==1.1.1" "evaluate==0.4.3" jiwer==3.0.5 bitsandbytes==0.44.1 soundfile librosa

In [ ]:
# 2. Upload sansa_training.zip
import os, zipfile
from google.colab import files
if not os.path.exists("data/train.jsonl"):
    up = files.upload()
    name = next(n for n in up if n.endswith(".zip"))
    zipfile.ZipFile(name).extractall("data")
print(open("data/README.txt").read())

In [ ]:
# 3. Hugging Face sign-in (paste your write token when asked)
from huggingface_hub import login, whoami
login()
USER = whoami()["name"]
print("Signed in as", USER)

In [ ]:
# 4. Load data and the current SANSA Tamil model
import json, torch, librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor, GenerationConfig
import os
BASE = os.environ.get("SANSA_BASE", "vasista22/whisper-tamil-medium")
GEN = os.environ.get("SANSA_GEN", "openai/whisper-medium")
proc = WhisperProcessor.from_pretrained(GEN, language="tamil", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(BASE)
model.generation_config = GenerationConfig.from_pretrained(GEN)
model.generation_config.language = "ta"; model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
def load(split):
    rows = [json.loads(l) for l in open(f"data/{split}.jsonl", encoding="utf-8")]
    for r in rows:
        r["audio"] = librosa.load(f"data/{r['file']}", sr=16000)[0]
    return rows
train, test = load("train"), load("test")
print(len(train), "training pieces,", len(test), "test pieces")

In [ ]:
# 5. Accuracy BEFORE training (on recordings the model never trains on)
import jiwer, re
def norm(s): return re.sub(r"[.,?!;:\-\"'()\[\]]", " ", s.lower()).split()
CUDA = torch.cuda.is_available()
def score(m):
    m.eval()
    if CUDA: m.cuda().half()
    hyps, refs = [], []
    for r in test:
        feats = proc(r["audio"], sampling_rate=16000, return_tensors="pt").input_features
        if CUDA: feats = feats.cuda().half()
        with torch.no_grad():
            ids = m.generate(feats, language="ta", task="transcribe", max_new_tokens=440)
        hyps.append(" ".join(norm(proc.batch_decode(ids, skip_special_tokens=True)[0])))
        refs.append(" ".join(norm(r["text"])))
    m.float()
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps), hyps[:3]
before = score(model)
print(f"BEFORE: word error {before[0]:.1%}, letter error {before[1]:.1%}")
for h, r in zip(before[2], test[:3]): print("  model :", h[:150]); print("  correct:", r["text"][:150])

In [ ]:
# 6. Train
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
class DS(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_features": proc.feature_extractor(r["audio"], sampling_rate=16000).input_features[0],
                "labels": proc.tokenizer(r["text"]).input_ids[:440]}
def collate(b):
    x = proc.feature_extractor.pad([{"input_features": f["input_features"]} for f in b], return_tensors="pt")
    y = proc.tokenizer.pad([{"input_ids": f["labels"]} for f in b], return_tensors="pt")
    labels = y["input_ids"].masked_fill(y.attention_mask.ne(1), -100)
    if (labels[:, 0] == model.config.decoder_start_token_id).all(): labels = labels[:, 1:]
    x["labels"] = labels
    return x
# Keep the sound part of the model mostly as it is (it already knows Tamil sounds); teach the rest your words.
for p in model.model.encoder.conv1.parameters(): p.requires_grad = False
for p in model.model.encoder.conv2.parameters(): p.requires_grad = False
for layer in model.model.encoder.layers[: len(model.model.encoder.layers) // 2]:
    for p in layer.parameters(): p.requires_grad = False
steps = max(30, min(1500, len(train) * 8 // 16))  # about 8 passes over your pieces
args = Seq2SeqTrainingArguments(output_dir="out", per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=1e-5, warmup_steps=max(3, steps // 10), max_steps=int(os.environ.get("SANSA_STEPS", steps)), fp16=CUDA,
    gradient_checkpointing=True, optim="adamw_bnb_8bit" if CUDA else "adamw_torch", logging_steps=20, save_strategy="no", report_to=[], remove_unused_columns=False,
    dataloader_num_workers=2)
model.config.use_cache = False
Seq2SeqTrainer(model=model, args=args, train_dataset=DS(train), data_collator=collate).train()
model.config.use_cache = True

In [ ]:
# 7. Accuracy AFTER training
after = score(model)
print(f"BEFORE: word error {before[0]:.1%}, letter error {before[1]:.1%}")
print(f"AFTER : word error {after[0]:.1%}, letter error {after[1]:.1%}")
for h, r in zip(after[2], test[:3]): print("  model :", h[:150]); print("  correct:", r["text"][:150])

In [ ]:
# 8. Save to your Hugging Face account (private) only if it got better
if after[0] < before[0]:
    repo = f"{USER}/whisper-tamil-medium-sansa"
    model.push_to_hub(repo, private=True)
    proc.push_to_hub(repo, private=True)
    print("DONE. Saved to https://huggingface.co/" + repo)
    print(f"Tell Claude: done, word error {before[0]:.1%} -> {after[0]:.1%}")
else:
    print("Training did not help this time, so nothing was saved. Tell Claude the BEFORE/AFTER numbers.")